코드 bertopic-korean

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
import pandas as pd

# 1) 한국어 데이터 로드
df = pd.read_csv('korean_news_8800.csv')
docs = df['body'].tolist()

# 2) 한국어 임베딩 모델 선택
embedding_model = SentenceTransformer(
    'jhgan/ko-sroberta-multitask'  # 한국어 SBERT 표준
)

# 3) UMAP과 HDBSCAN 설정
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42       # 재현가능성
)
hdbscan_model = HDBSCAN(
    min_cluster_size=30,  # 최소 토픽 크기
    metric='euclidean',
    cluster_selection_method='eom'
)

# 4) BERTopic 모델 구성과 학습
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language='multilingual',
    calculate_probabilities=True,
    verbose=True
)
topics, probs = topic_model.fit_transform(docs)

# 5) 토픽 정보 출력
print(topic_model.get_topic_info().head(20))
# 토픽 5번의 상위 키워드
print(topic_model.get_topic(5))

코드 matthes-kohring-prompt

In [ ]:
MATTHES_KOHRING_PROMPT = """
당신은 미디어 프레임 분석 전문가입니다.
Matthes & Kohring(2008)의 5요소 인덕티브 접근에 따라
주어진 기사에서 다음 5개 프레임 요소를 추출하세요.

## 5개 프레임 요소 (Entman, 1993의 정의 기반)

### 1. problem (문제 정의)
- 기사가 무엇을 '문제'로 진단하는가?
- 한 문장으로 추출 (40자 이내)
- 명시적이지 않으면 "unclear"

### 2. cause (원인 진단)
- 그 문제의 원인을 누구/무엇으로 진단하는가?
- 한 문장 + 주체(persons/orgs) 명시

### 3. moral (도덕적 평가)
- 누가 옳고 누가 그른가? 평가는 어디서 비롯되나?
- 한 문장 + 평가 톤(positive/negative/neutral)
- 평가가 없으면 "none"

### 4. solution (해결 권고)
- 어떤 해결책·대응책이 제시되는가?
- 한 문장으로 추출. 없으면 "none"

### 5. actors (관련 행위자)
- 등장 행위자와 그 역할
- 배열 형식: name, role(protagonist/antagonist/neutral),
  type(person/organization)

## 메타데이터
- confidence: 추출의 확신도 (0.0~1.0)
- text_basis: 추출의 핵심 근거 문장 (원문 인용, 최대 2개)

## 출력 형식
반드시 유효한 JSON으로만 출력하세요.
다른 설명 없이 JSON 객체만 출력하세요.

## 중요 원칙
1. '추출'이지 '분류'가 아닙니다. 텍스트에 명시된 것만 추출하세요.
2. 한국어로 추출하되, 핵심 행위자명은 원문 표기를 유지하세요.
3. 불확실하면 confidence를 낮게 설정하고 "unclear"/"none"을 사용하세요.
4. 사설·칼럼의 경우 필자의 평가를 "moral"에,
   인용된 외부 평가를 "actors" 역할로 구분하세요.
5. 추측·연역을 하지 말고 텍스트가 실제로 제시한 것만 추출하세요.
"""

코드 mcp-server-pattern

In [ ]:
from fastmcp import FastMCP
import json
from anthropic import AsyncAnthropic

mcp = FastMCP("Inductive Frame Analysis")
client = AsyncAnthropic()

@mcp.tool()
async def extract_frame_elements(article_text: str, article_id: str) -> dict:
    """기사에서 Matthes & Kohring 5요소를 추출한다."""
    response = await client.messages.create(
        model="claude-opus-4-7",
        max_tokens=1500,
        system=MATTHES_KOHRING_PROMPT,
        messages=[{
            "role": "user",
            "content": f"기사 ID: {article_id}\n\n{article_text[:3000]}"
        }],
        temperature=0
    )
    return json.loads(response.content[0].text)

@mcp.tool()
async def cluster_elements(elements: list, dim: str = "problem") -> dict:
    """추출된 특정 요소를 BERTopic으로 클러스터링한다."""
    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer

    embedding_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
    topic_model = BERTopic(embedding_model=embedding_model,
                            language='multilingual',
                            min_topic_size=20)
    texts = [e[dim] for e in elements if e[dim] not in ("unclear", "none")]
    topics, _ = topic_model.fit_transform(texts)
    return {
        "topics": topic_model.get_topic_info().to_dict('records'),
        "topic_assignments": list(zip([e['article_id'] for e in elements], topics))
    }

@mcp.tool()
async def calculate_reliability(llm_labels: list, human_labels: list,
                                  dim: str = "problem") -> dict:
    """LLM과 인간 코더의 일치도를 Krippendorff α로 계산한다."""
    import krippendorff
    # ... (5장 5.5.2절 패턴)
    return {"alpha": ..., "ci_95": [..., ...], "n": len(llm_labels)}

if __name__ == "__main__":
    mcp.run()

코드 lora-finetuning

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import torch

# 1) 기반 모델 (한국어 BERT)
model_name = "klue/roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=4  # 진보·중도·보수·중립
)

# 2) LoRA 설정
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                # 저차원 행렬의 rank
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["query", "value"]  # 어텐션의 Q, V만 학습
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 출력: "trainable params: 1.2M || all params: 337M || trainable%: 0.36"

# 3) 학습 (인간 코더 라벨링 4,000건 데이터 가정)
training_args = TrainingArguments(
    output_dir="./korean-media-classifier",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    save_strategy="epoch",
    fp16=True
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer
)
trainer.train()

# 4) 저장과 추론 (어댑터만 저장, 단 수 MB)
model.save_pretrained("./lora-adapter")

코드 llm-interpretability

In [ ]:
import asyncio
from anthropic import AsyncAnthropic

client = AsyncAnthropic()

async def analyze_with_perturbation(article_text: str, paragraphs: list):
    """기사의 각 문단을 제거한 뒤 LLM의 추출 변화를 측정."""

    # 원본 분석
    original = await extract_frame_elements(article_text)

    # 각 문단을 제거한 분석
    influences = []
    for i in range(len(paragraphs)):
        modified_text = "\n\n".join(
            p for j, p in enumerate(paragraphs) if j != i
        )
        modified = await extract_frame_elements(modified_text)
        # 각 차원에서 변화 정도 계산
        diff = {
            'paragraph_idx': i,
            'paragraph': paragraphs[i][:100],
            'problem_changed': original['problem'] != modified['problem'],
            'moral_changed': original['moral'] != modified['moral'],
            'actors_diff': set(a['name'] for a in original['actors']) ^
                            set(a['name'] for a in modified['actors'])
        }
        influences.append(diff)

    return influences

# 사용: ‘이 기사의 LLM 분석에 가장 영향력 있는 문단은?’
important = sorted(influences,
                    key=lambda x: sum([x['problem_changed'],
                                        x['moral_changed']]),
                    reverse=True)[:3]